# Patient Monitoring - Kafka Ingestion Pipeline

This notebook demonstrates:

1. Kafka producer
2. Kafka consumer
3. Pydantic schema validation
4. Invalid records routing to Dead Letter Queue (DLQ)

Architecture:

Producer
    |
    v
Kafka Topic
    |
    v
Consumer
    |
    +--> Valid Records
    |
    +--> DLQ (invalid records + rejection reason)

## Imports

In [1]:
import json

from pydantic import BaseModel, ValidationError
from kafka import KafkaProducer, KafkaConsumer

## Schema Validation

Pydantic is used as the ingestion boundary.
Only records matching the contract are accepted.

## Data Contract

In [2]:
class PatientReading(BaseModel):
    patient_id: str
    heart_rate: int
    oxygen_level: int

## Kafka Producer Setup

In [3]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

## Generate Test Data

In [4]:
valid_record = {
    "patient_id": "P001",
    "heart_rate": 82,
    "oxygen_level": 98
}


invalid_record = {
    "patient_id": "P002",
    "heart_rate": "HIGH",
    "oxygen_level": 97
}


producer.send(
    "patient_readings",
    valid_record
)

producer.send(
    "patient_readings",
    invalid_record
)

producer.flush()

print("Messages sent")

Messages sent


## consumer Setup

In [5]:
consumer = KafkaConsumer(
    "patient_readings",
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    consumer_timeout_ms=5000
)

C:\Users\reems\AppData\Local\Temp\ipykernel_17008\3076173257.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


## DLQ Producer

In [6]:
dlq_producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

## Validation Boundary

In [7]:
valid_records = []


for msg in consumer:

    record = msg.value

    try:

        validated = PatientReading(**record)

        valid_records.append(
            validated.model_dump()
        )

        print("VALID:", record)


    except ValidationError as e:

        dlq_message = {
            "record": record,
            "reason": str(e)
        }

        dlq_producer.send(
            "patient_readings_dlq",
            dlq_message
        )

        print("INVALID:", record)


dlq_producer.flush()

VALID: {'patient_id': 'P001', 'heart_rate': 82, 'oxygen_level': 98}
INVALID: {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}
VALID: {'patient_id': 'P001', 'heart_rate': 82, 'oxygen_level': 98}
INVALID: {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}


## DLQ Evidence

In [8]:
dlq_consumer = KafkaConsumer(
    "patient_readings_dlq",
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    consumer_timeout_ms=5000
)


for msg in dlq_consumer:
    print(msg.value)

C:\Users\reems\AppData\Local\Temp\ipykernel_17008\233827345.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  dlq_consumer = KafkaConsumer(


{'record': {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}, 'reason': "1 validation error for PatientReading\nheart_rate\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='HIGH', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing"}
{'record': {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}, 'reason': "1 validation error for PatientReading\nheart_rate\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='HIGH', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing"}
{'record': {'patient_id': 'P002', 'heart_rate': 'HIGH', 'oxygen_level': 97}, 'reason': "1 validation error for PatientReading\nheart_rate\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='HIGH', input_type=str]\n    For further information visit